Create CQL libraries for US-Core

1. Download the latest java cli CQF tool `tooling-cli-<version>.jar` from `https://central.sonatype.com/artifact/org.opencds.cqf/tooling-cli` directly via your browser using the Maven Central URL:

     e.g., for version 3.11.0:  `https://repo1.maven.org/maven2/org/opencds/cqf/tooling-cli/3.11.0/tooling-cli-3.11.0.jar`

   - Update `jarVersion` in the `%%bash` cell to match the jar downloaded.

2. Set the version in the config cell. `VERSIONS` there is the single source of truth: the `%%bash` cell reads it from the kernel environment and the python cell reads it directly, so it is only set in one place. The string has to match the `FHIR-Spec/US-Core/<version>` folder name exactly (e.g., "10.0.0-ballot").

3. Download and unpack the IG definitions file for each version into `/Users/ehaas/Documents/Python/Jupyter/MyNotebooks/bash_scripts/FHIR-Spec/US-Core` as `<version>`:
   - For a published version, download `https://hl7.org/fhir/us/core/STU<N>/definitions.json.zip`
   - For a version still in development, use the local build output at `/Users/ehaas/Documents/FHIR/US-Core/output/definitions.json.zip`, which the IG Publisher rewrites on every build

4. Run the `%%bash` cell to generate the ModelInfo. It is saved in the `bash_scripts/output` folder as lowercase `<modelName>-modelinfo-<modelVersion>.xml`

5. Run the python cell to package the ModelInfo into a `Library` resource, convert it to YAML, and write it to `/Users/ehaas/Documents/FHIR/US-Core/input/examples-yaml/library-uscore-modelinfo.yml`

6. In the US Core repo, run `./publish.sh -y` to regenerate `input/examples/library-uscore-modelinfo.json` from the new YAML

Only the current version is generated. US Core removed the prior versions' Library resources in 9.0.0 ([FHIR-54705](https://jira.hl7.org/browse/FHIR-54705)); the commented-out list in the config cell regenerates all of them if ever needed:
  - US Core Version 3.1.1 Model Definition
  - US Core Version 4.0.0 Model Definition
  - US Core Version 5.0.1 Model Definition
  - US Core Version 6.1.0 Model Definition
  - US Core Version 7.0.0 Model Definition
  - US Core Version 8.0.1 Model Definition
  - US Core Version 9.0.0 Model Definition
  - US Core Version 10.0.0-ballot Model Definition

In [5]:
# ==== single source of truth for the version(s) being generated ====
# Must match the FHIR-Spec/US-Core/<version> folder name(s).
# Run this cell before the %%bash cell below: the bash cell reads VERSIONS from
# the kernel's environment, so a restarted kernel needs this cell re-run.
import os

# VERSIONS = ["3.1.1", "4.0.0", "5.0.1", "6.1.0", "7.0.0", "8.0.1", "9.0.0"]  # multiple prior versions
VERSIONS = ["10.0.0-ballot"]

os.environ["VERSIONS"] = " ".join(VERSIONS)
print(f'{VERSIONS = }')

VERSIONS = ['10.0.0-ballot']


In [6]:
%%bash
# create a ModelInfo xml document
set -euo pipefail
: "${VERSIONS:?run the config cell above first}"

jarVersion="3.11.0"
inputPath="/Users/ehaas/Documents/Python/Jupyter/MyNotebooks/bash_scripts/FHIR-Spec"
modelName="USCore"

# $VERSIONS is deliberately unquoted here: word splitting turns it back into a list
for VER in $VERSIONS; do
echo "$VER"

resourcePaths="4.0.1;US-Core/${VER}"
modelVersion="$VER"

echo "java  -jar /Users/ehaas/Downloads/tooling-cli-$jarVersion.jar -GenerateMIs -ip=$inputPath -rp=$resourcePaths -mn=$modelName -mv=$modelVersion -im=false -ucp=true"
java  -jar /Users/ehaas/Downloads/tooling-cli-$jarVersion.jar -GenerateMIs -ip=$inputPath -rp=$resourcePaths -mn=$modelName -mv="$modelVersion" -im=false -ucp=true
done

10.0.0-ballot
java  -jar /Users/ehaas/Downloads/tooling-cli-3.11.0.jar -GenerateMIs -ip=/Users/ehaas/Documents/Python/Jupyter/MyNotebooks/bash_scripts/FHIR-Spec -rp=4.0.1;US-Core/10.0.0-ballot -mn=USCore -mv=10.0.0-ballot -im=false -ucp=true


[main] INFO org.opencds.cqf.tooling.modelinfo.Atlas - Reading 4.0.1 Conformance Resources
[main] INFO ca.uhn.fhir.util.VersionUtil - HAPI FHIR version 8.0.0 - Rev 091beb6d18
[main] INFO ca.uhn.fhir.context.FhirContext - Creating new FHIR context for FHIR version [R4]
[main] INFO ca.uhn.fhir.util.XmlUtil - Unable to determine StAX implementation: java.xml/META-INF/MANIFEST.MF not found
[main] INFO org.opencds.cqf.tooling.modelinfo.Atlas - Duplicate CodeSystem with id null
[main] INFO org.opencds.cqf.tooling.modelinfo.Atlas - Duplicate CodeSystem with id null
[main] INFO org.opencds.cqf.tooling.modelinfo.Atlas - Duplicate url found for: http://hl7.org/fhir/StructureDefinition/de-Quantity.value
[main] INFO org.opencds.cqf.tooling.modelinfo.Atlas - Duplicate url found for: http://hl7.org/fhir/StructureDefinition/de-Quantity.comparator
[main] INFO org.opencds.cqf.tooling.modelinfo.Atlas - Duplicate url found for: http://hl7.org/fhir/StructureDefinition/de-Quantity.unit
[main] INFO org.openc

In [7]:
from datetime import datetime
from pathlib import Path
import base64
from yaml import dump

versions = VERSIONS  # === set in the config cell at the top ===
yaml_comment = """
#****************************************************************************************************
#                                   WARNING: DO NOT EDIT THIS FILE                                  *
#                                                                                                   *
# This file is generated by a Jupyter Script at                                                     *
# https://github.com/Healthedata1/MyNotebooks/blob/master/bash_scripts/create_cql_libraries.ipynb   *
# Any edits you make to this file will be overwritten.                                              *
#                                                                                                   *
# To change the contents of this file, edit the script file directly                                *
#****************************************************************************************************
"""
today = str(datetime.now().date())

def xml_to_base64(file_path):
    try:
        # Create Path object
        xml_file = Path(file_path)
        # Read the XML file in binary mode
        xml_content = xml_file.read_bytes()
        # Convert to Base64
        base64_string = base64.b64encode(xml_content).decode('utf-8')
        return base64_string
    except FileNotFoundError:
        return "Error: XML file not found"
    except Exception as e:
        return f"Error: {str(e)}"



'''
To ensure the model information can be loaded from the published IG package:

id is USCore-ModelInfo
Name is USCore
URL is http://hl7.org/fhir/Library/USCore-ModelInfo
Name is USCore
Title is US Core Version {ver} Model Definition
Version is USCore Version

To avoid collisions between model info names and library names, the implementation will look for a library
whose canonical ends in USCore-ModelInfo, so we put that in a conformance requirement
in the Using CQL With FHIR IG: https://hl7.org/fhir/uv/cql/using-modelinfo.html#conformance-requirement-6-
'''

for ver in versions:
    file_path = f'/Users/ehaas/Documents/Python/Jupyter/MyNotebooks/bash_scripts/output/uscore-modelinfo-{ver}.xml'
    out_path = Path(f'/Users/ehaas/Documents/FHIR/US-Core/input/examples-yaml/library-uscore-modelinfo.yml')
    library = {
              "resourceType" : "Library",
              "id" : "USCore-ModelInfo",
              # ==== The profile leads to publisher error
              # "meta" : {
              #   "profile" : ["http://hl7.org/fhir/uv/cql/StructureDefinition/cql-modelinfo"]
              # },
              "url" : "http://hl7.org/fhir/Library/USCore-ModelInfo",
              "name" : f"USCore",
              "title" : f"US Core Model Definition",
              "status" : "active",
              "experimental" : False,
              "type" : {
                "coding" : [{
                  "system" : "http://terminology.hl7.org/CodeSystem/library-type",
                  "code" : "model-definition"
                }]
              },
              "date" : today,
              "description" : f"The CQL Model Information for US Core Version {ver}",
              "lastReviewDate" : today,
              "topic" : [{
                "text" : f"US Core Version {ver}"
              }],
              "content" : [{
                "contentType" : "application/xml",
                "data" : f"{xml_to_base64(file_path)}"
              }]
            }
    example_yaml = f'{yaml_comment}\n{dump(library, indent=2, sort_keys=False)}'
    print(f'{"="*20} version = {ver} {"="*20}')
    print(example_yaml)
    print(f'{"="*80}')
    print(f'writing to {out_path}')
    out_path.write_text(example_yaml)
print("done!")

==================== version = 10.0.0-ballot ====================

#****************************************************************************************************
#                                   WARNING: DO NOT EDIT THIS FILE                                  *
#                                                                                                   *
# This file is generated by a Jupyter Script at                                                     *
# https://github.com/Healthedata1/MyNotebooks/blob/master/bash_scripts/create_cql_libraries.ipynb   *
# Any edits you make to this file will be overwritten.                                              *
#                                                                                                   *
# To change the contents of this file, edit the script file directly                                *
#****************************************************************************************************

resourceType: 